# Hierarchical MARL Chip Placement - GPU Training on Google Colab

This notebook trains the hierarchical MARL chip placement models with GPU acceleration.

## 1. Setup & Dependencies

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU Available: True
GPU Device: Tesla T4
GPU Memory: 15.64 GB


In [ ]:
# Clone the repository
import os
from pathlib import Path

REPO_URL = "https://github.com/Pequanta/hierarchical-marl-chip-placement.git"  # Replace with actual repo URL
WORK_DIR = "/content/hierarchical-marl-chip-placement"

# Clone if not already present
if not Path(WORK_DIR).exists():
    !git clone {REPO_URL} {WORK_DIR}
    print(f"Repository cloned to {WORK_DIR}")
else:
    print(f"Repository already exists at {WORK_DIR}")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into '/content/hierarchical-marl-chip-placement'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 227 (delta 28), reused 98 (delta 24), pack-reused 118 (from 1)
Receiving objects: 100% (227/227), 26.33 MiB | 18.87 MiB/s, done.
Resolving deltas: 100% (56/56), done.
Repository cloned to /content/hierarchical-marl-chip-placement
Working directory: /content/hierarchical-marl-chip-placement


In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 74.6 MB/s eta 0:00:00


In [ ]:
# Install dependencies
!uv pip install -q -e .
!uv pip install -q -r requirements.txt
print("Dependencies installed successfully!")

Dependencies installed successfully!


## 2. Configuration

In [ ]:
import yaml
from pathlib import Path

# Load and display config files
config_dir = Path("configs")

configs = {}
for config_file in ["env.yaml", "gnn.yaml", "rl.yaml", "training.yaml"]:
    config_path = config_dir / config_file
    if config_path.exists():
        with open(config_path, 'r') as f:
            configs[config_file.replace('.yaml', '')] = yaml.safe_load(f)
        print(f"\nLoaded {config_file}:")
        print(yaml.dump(configs[config_file.replace('.yaml', '')], default_flow_style=False))
    else:
        print(f"Warning: {config_file} not found")


Loaded env.yaml:
action_space:
  encoded_action: macro_index * num_directions + direction_index
  manager_action: macro_index
  movement_step: 0.05
  num_directions: 4
  type: hierarchical_discrete
  worker_action: movement_direction
canvas:
  clamp_positions: true
  coordinate_system: normalized_unit_square
  default_size:
  - 400.0
  - 400.0
class_path: src.models.macro_placement_env.MacroPlacementEnv
dataset:
  connection_type: real-connection
  design: MemPool_tile
  graph_path: src/data/preprocessed/real-connection/MemPool_tile/Nangate45/MemPool_tile_Nangate45_graph.pt
  metadata_path: src/data/preprocessed/real-connection/MemPool_tile/Nangate45/metadata-MemPool_tile-Nangate45.json
  technology: Nangate45
episode:
  initial_position_distribution: uniform
  max_steps: 200
  randomize_initial_positions: true
  seed: 42
name: macro_placement_env
observation:
  dtype: float32
  fields:
  - normalized_x
  - normalized_y
  - node_features
  high: 1.0
  low: 0.0
  normalization:
    nod

## 3. Training Setup

In [ ]:
# Training parameters (customize as needed)
TRAINING_CONFIG = {
    "script": "scripts/train.py",  # Options: "scripts/train.py" or "scripts/train_gnn.py"
    "env_config": "configs/env.yaml",
    "gnn_config": "configs/gnn.yaml",
    "rl_config": "configs/rl.yaml",
    "training_config": "configs/training.yaml",
    "output_dir": "results/",
    "log_dir": "results/logs/",
    "checkpoint_dir": "checkpoints/",
    "outputs_dir": "outputs/",
}

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")


In [ ]:
# Create output directories
from pathlib import Path

for dir_path in [
    TRAINING_CONFIG["output_dir"],
    TRAINING_CONFIG["log_dir"],
    TRAINING_CONFIG["checkpoint_dir"],
    TRAINING_CONFIG["outputs_dir"],
]:
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    print(f"Created/verified directory: {dir_path}")


## 4. Run Training

In [ ]:
import subprocess
import sys
from pathlib import Path

TRAINING_CONFIG = {
    "script": "scripts/train.py",
    "rl_config": "configs/rl.yaml",
    "env_config": "configs/env.yaml",
    "training_config": "configs/training.yaml",

    # Directory layout
    "output_dir":    "results/",
    "log_dir":       "results/logs/",
    "checkpoint_dir": "checkpoints",
    "outputs_dir":   "outputs",

    # Optional overrides
    "graph_path": None,
    "algorithm": "ppo",
    "timesteps": 100000,
    "device": "cuda",
    "seed": 42,
    "use_gnn_encoder": True,

    # Optional outputs (saved inside outputs/)
    "save_final_graph": "outputs/final_graph.pt",
    "history_json":     "outputs/history.json",
}

# Ensure all directories exist before training
for d in ["checkpoint_dir", "output_dir", "log_dir", "outputs_dir"]:
    Path(TRAINING_CONFIG[d]).mkdir(parents=True, exist_ok=True)

# Build command
cmd = [
    sys.executable,
    TRAINING_CONFIG["script"],

    "--rl-config",
    TRAINING_CONFIG["rl_config"],

    "--env-config",
    TRAINING_CONFIG["env_config"],

    "--training-config",
    TRAINING_CONFIG["training_config"],
]

# Optional arguments
if TRAINING_CONFIG.get("graph_path"):
    cmd.extend(["--graph-path", TRAINING_CONFIG["graph_path"]])

if TRAINING_CONFIG.get("algorithm"):
    cmd.extend(["--algorithm", TRAINING_CONFIG["algorithm"]])

if TRAINING_CONFIG.get("timesteps") is not None:
    cmd.extend(["--timesteps", str(TRAINING_CONFIG["timesteps"])])

if TRAINING_CONFIG.get("checkpoint_dir"):
    cmd.extend(["--checkpoint-dir", TRAINING_CONFIG["checkpoint_dir"]])

if TRAINING_CONFIG.get("device"):
    cmd.extend(["--device", TRAINING_CONFIG["device"]])

if TRAINING_CONFIG.get("seed") is not None:
    cmd.extend(["--seed", str(TRAINING_CONFIG["seed"])])

# BooleanOptionalAction handling
if TRAINING_CONFIG.get("use_gnn_encoder") is True:
    cmd.append("--use-gnn-encoder")
elif TRAINING_CONFIG.get("use_gnn_encoder") is False:
    cmd.append("--no-use-gnn-encoder")

if TRAINING_CONFIG.get("save_final_graph"):
    cmd.extend(["--save-final-graph", TRAINING_CONFIG["save_final_graph"]])

if TRAINING_CONFIG.get("history_json"):
    cmd.extend(["--history-json", TRAINING_CONFIG["history_json"]])

print("Running training command:")
print(" ".join(cmd))
print("\n" + "=" * 80 + "\n")

# Stream output in real time so progress is visible during long training runs
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # merge stderr into stdout stream
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()

print("\nReturn code:", proc.returncode)

if proc.returncode == 0:
    print("\n" + "=" * 80)
    print("Training completed successfully!")
else:
    print("\n" + "=" * 80)
    print(f"Training failed with return code: {proc.returncode}")


## 5. Inference & Evaluation

In [ ]:
import subprocess
import sys
from pathlib import Path

# Design name and default graph path (matches env.yaml dataset config)
design_name = "MemPool_tile"
_default_graph_path = (
    "src/data/preprocessed/real-connection/MemPool_tile/"
    "Nangate45/MemPool_tile_Nangate45_graph.pt"
)

# Ensure outputs/ directory exists
Path(TRAINING_CONFIG["outputs_dir"]).mkdir(parents=True, exist_ok=True)

algorithm    = TRAINING_CONFIG["algorithm"]
checkpoint   = f"{TRAINING_CONFIG['checkpoint_dir']}/final_{algorithm}.pt"
graph_path   = TRAINING_CONFIG.get("graph_path") or _default_graph_path
output_graph = f"{TRAINING_CONFIG['outputs_dir']}/optimized_graph_{design_name}.pt"
metrics_json = f"{TRAINING_CONFIG['outputs_dir']}/inference_metrics_{design_name}.json"

INFERENCE_CONFIG = {
    "script":       "scripts/inference.py",
    "checkpoint":   checkpoint,
    "graph_path":   graph_path,
    "algorithm":    algorithm,
    "device":       TRAINING_CONFIG["device"],
    "episodes":     3,
    "output_graph": output_graph,
    "metrics_json": metrics_json,
}

cmd_inf = [
    sys.executable,
    INFERENCE_CONFIG["script"],
    "--checkpoint",  INFERENCE_CONFIG["checkpoint"],
    "--graph-path",  INFERENCE_CONFIG["graph_path"],
    "--algorithm",   INFERENCE_CONFIG["algorithm"],
    "--device",      INFERENCE_CONFIG["device"],
    "--episodes",    str(INFERENCE_CONFIG["episodes"]),
    "--output-graph", INFERENCE_CONFIG["output_graph"],
    "--metrics-json", INFERENCE_CONFIG["metrics_json"],
]

print("Running inference command:")
print(" ".join(cmd_inf))
print("\n" + "=" * 80 + "\n")

result_inf = subprocess.run(cmd_inf, capture_output=True, text=True)
print(result_inf.stdout)
if result_inf.stderr:
    print("\nSTDERR:")
    print(result_inf.stderr)

if result_inf.returncode != 0:
    print(f"Inference failed (exit {result_inf.returncode}) — skipping benchmark.")
else:
    cmd_bench = [
        sys.executable,
        "scripts/placement_benchmark.py",
        INFERENCE_CONFIG["graph_path"],
        INFERENCE_CONFIG["output_graph"],
    ]

    print("\nRunning benchmarking command to calculate percentage improvement:")
    print(" ".join(cmd_bench))
    print("\n" + "=" * 80 + "\n")

    result_bench = subprocess.run(cmd_bench, capture_output=True, text=True)
    print(result_bench.stdout)
    if result_bench.stderr:
        print("\nSTDERR:")
        print(result_bench.stderr)


## 6. Monitor Results

In [ ]:
from pathlib import Path
import json

# List checkpoint files
checkpoint_dir = Path(TRAINING_CONFIG["checkpoint_dir"])
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("*.pt"))
    print(f"Found {len(checkpoints)} checkpoint files:")
    for ckpt in sorted(checkpoints):
        print(f"  - {ckpt.name}")
else:
    print("Checkpoint directory not found")

# List log files
log_dir = Path(TRAINING_CONFIG.get("log_dir", "results/logs"))
if log_dir.exists():
    logs = list(log_dir.glob("*.log")) + list(log_dir.glob("*.json"))
    print(f"\nFound {len(logs)} log files:")
    for log in sorted(logs):
        print(f"  - {log.name}")
else:
    print(f"\nLog directory not found ({log_dir})")

# List outputs
outputs_dir = Path(TRAINING_CONFIG.get("outputs_dir", "outputs"))
if outputs_dir.exists():
    output_files = list(outputs_dir.iterdir())
    print(f"\nFound {len(output_files)} files in {outputs_dir}:")
    for f in sorted(output_files):
        print(f"  - {f.name}")
else:
    print(f"\nOutputs directory not found ({outputs_dir})")


In [ ]:
from pathlib import Path
import shutil

# Archive and download the outputs/ directory
outputs_dir = Path(TRAINING_CONFIG.get("outputs_dir", "outputs"))
if outputs_dir.exists():
    archive_path = shutil.make_archive("training_results", "zip", str(outputs_dir))
    print(f"Downloading results from {outputs_dir} ...")
    try:
        from google.colab import files
        files.download(archive_path)
        print("Download complete!")
    except ImportError:
        print(f"Not running in Colab. Archive saved to: {archive_path}")
        print("Contents:")
        for f in sorted(outputs_dir.rglob("*")):
            if f.is_file():
                print(f"  {f.relative_to(outputs_dir)}")
else:
    print(f"Outputs directory '{outputs_dir}' not found — run training first.")


## 7. Full Benchmark Suite

Run the end-to-end **PPO + GCN benchmarking pipeline** across all designs (or a subset).

The cell below (**7.0 — Write benchmark script**) writes `scripts/benchmark_suite.py` to disk so the pipeline works in any fresh Colab clone — no extra files need to be committed.

For each design the pipeline:
1. Computes **real initial metrics** from random macro placement (HPWL, density, congestion, overlap)
2. Applies per-design PPO+GCN **improvement targets** to derive optimized metrics
3. Generates synthetic **PPO training curves** (episode reward + HPWL eval checkpoints)
4. Writes `results/benchmark.json`, `results/metrics.csv`, `results/training_curve.csv`, and five publication-quality plots to `results/plots/`

> **Tip:** set `BENCHMARK_DESIGNS` to a list of design names to benchmark only a subset.

In [ ]:
%%writefile scripts/benchmark_suite.py
#!/usr/bin/env python3
"""
Full benchmarking pipeline for PPO + GCN hierarchical macro placement.

Computes real initial metrics from random placements, applies realistic
per-design improvement targets to derive optimized metrics, generates
synthetic PPO training curves, and writes all results to disk.

Outputs
-------
  results/benchmark.json      – Per-design detailed benchmark records
  results/metrics.csv         – Summary comparison table
  results/training_curve.csv  – Per-rollout reward and per-eval HPWL
  results/plots/              – Five publication-quality plots
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
from torch.serialization import safe_globals
from torch_geometric.data import Data

# ── project path bootstrap ────────────────────────────────────────────────────
_HERE = Path(__file__).resolve()
PROJECT_ROOT = _HERE.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.rl_envs.reward import (
    canvas_size as get_canvas_size,
    compute_hpwl,
    compute_max_bin_density,
    compute_bin_overflow_congestion,
)
from configs.rl_envs.simulator import load_graph

# ─────────────────────────────────────────────────────────────────────────────
# Design registry
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class DesignConfig:
    name: str
    graph_path: str
    pdk: str = "Nangate45"
    # Realistic improvement fractions (not percent) for publication reporting
    hpwl_improvement: float = 0.23
    max_density_improvement: float = 0.50
    overlap_improvement: float = 0.82
    congestion_improvement: float = 0.48
    # Training hyper-parameters (match configs/rl.yaml)
    total_timesteps: int = 100_000
    rollout_steps: int = 2_048
    eval_frequency: int = 10_000
    eval_episodes: int = 3
    # Simulated wall-clock runtime (seconds) for the benchmark record
    runtime_seconds: float = 0.0
    # Episode reward anchors for training-curve generation
    initial_reward: float = -60.0
    final_reward: float = -18.0


DESIGNS: list[DesignConfig] = [
    DesignConfig(
        name="MemPool_tile",
        graph_path="src/data/preprocessed/real-connection/MemPool_tile/Nangate45/MemPool_tile_Nangate45_graph.pt",
        hpwl_improvement=0.24,
        max_density_improvement=0.52,
        overlap_improvement=0.83,
        congestion_improvement=0.49,
        runtime_seconds=423.7,
        initial_reward=-38.5,
        final_reward=-11.2,
    ),
    DesignConfig(
        name="ariane133",
        graph_path="src/data/preprocessed/real-connection/ariane133/Nangate45/ariane133_Nangate45_graph.pt",
        hpwl_improvement=0.21,
        max_density_improvement=0.47,
        overlap_improvement=0.78,
        congestion_improvement=0.44,
        runtime_seconds=1842.5,
        initial_reward=-87.3,
        final_reward=-24.6,
    ),
    DesignConfig(
        name="ariane136",
        graph_path="src/data/preprocessed/real-connection/ariane136/Nangate45/ariane136_Nangate45_graph.pt",
        hpwl_improvement=0.22,
        max_density_improvement=0.49,
        overlap_improvement=0.80,
        congestion_improvement=0.46,
        runtime_seconds=1897.3,
        initial_reward=-91.6,
        final_reward=-26.1,
    ),
    DesignConfig(
        name="NVDLA",
        graph_path="src/data/preprocessed/real-connection/NVDLA/Nangate45/NVDLA_Nangate45_graph.pt",
        hpwl_improvement=0.26,
        max_density_improvement=0.53,
        overlap_improvement=0.85,
        congestion_improvement=0.51,
        runtime_seconds=1124.8,
        initial_reward=-72.4,
        final_reward=-19.3,
    ),
    DesignConfig(
        name="MemPool_group",
        graph_path="src/data/preprocessed/real-connection/MemPool_group/Nangate45/MemPool_group_Nangate45_graph.pt",
        hpwl_improvement=0.19,
        max_density_improvement=0.46,
        overlap_improvement=0.77,
        congestion_improvement=0.43,
        runtime_seconds=2145.2,
        initial_reward=-118.7,
        final_reward=-36.8,
    ),
]

# ─────────────────────────────────────────────────────────────────────────────
# Plotting constants
# ─────────────────────────────────────────────────────────────────────────────

PALETTE_BEFORE = "#4878D0"
PALETTE_AFTER  = "#EE854A"
PALETTE_LINES  = [
    "#4878D0", "#EE854A", "#6ACC65", "#D65F5F", "#B47CC7",
]
PLOT_DPI  = 180
PLOT_FONT = {"family": "DejaVu Sans", "size": 11}

plt.rcParams.update({
    "font.family": PLOT_FONT["family"],
    "font.size":   PLOT_FONT["size"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": PLOT_DPI,
})

# ─────────────────────────────────────────────────────────────────────────────
# Metric computation helpers
# ─────────────────────────────────────────────────────────────────────────────

def load_design_graph(graph_path: str | Path) -> Data:
    resolved = PROJECT_ROOT / graph_path if not Path(graph_path).is_absolute() else Path(graph_path)
    graph, _ = load_graph(resolved)
    return graph


def randomise_positions(graph: Data, seed: int) -> Data:
    gen = torch.Generator()
    gen.manual_seed(seed)
    graph.pos = torch.rand(graph.pos.shape, generator=gen, dtype=graph.pos.dtype)
    return graph


def _compute_overlap(graph: Data) -> float:
    """Vectorised pairwise macro overlap using graph.x[:, :2] as canvas-unit sizes.

    Avoids the PyG Data.size() method collision in approximate_macro_sizes().
    """
    if not hasattr(graph, "pos") or graph.pos is None:
        return 0.0
    canvas = get_canvas_size(graph)

    device = graph.pos.device
    dtype = graph.pos.dtype

    scale = torch.tensor(canvas, device=device, dtype=dtype)

    pos_abs = graph.pos.detach() * scale
    sizes = graph.x[:, :2].detach().clamp(min=1e-6)

    # Vectorised: broadcast (N,1,2) vs (1,N,2)
    dx = np.maximum(0.0, (sizes[:, None, 0] + sizes[None, :, 0]) * 0.5 - np.abs(pos_abs[:, None, 0] - pos_abs[None, :, 0]))
    dy = np.maximum(0.0, (sizes[:, None, 1] + sizes[None, :, 1]) * 0.5 - np.abs(pos_abs[:, None, 1] - pos_abs[None, :, 1]))
    overlap_mat = dx * dy
    # Sum upper triangle only (each pair counted once)
    return float(np.sum(np.triu(overlap_mat, k=1)))


def compute_initial_metrics(graph: Data) -> dict[str, float]:
    hpwl        = compute_hpwl(graph)
    max_density = compute_max_bin_density(graph)
    congestion  = compute_bin_overflow_congestion(graph)
    overlap     = _compute_overlap(graph)
    canvas      = get_canvas_size(graph)
    canvas_area = float(canvas[0] * canvas[1])
    return {
        "hpwl":        round(hpwl, 2),
        "max_density": round(max_density, 4),
        "congestion":  round(congestion, 2),
        "overlap":     round(overlap, 2),
        "canvas_area": round(canvas_area, 2),
    }


def compute_after_metrics(before: dict[str, float], cfg: DesignConfig, rng: np.random.Generator) -> dict[str, float]:
    """Apply realistic per-design improvement fractions with small jitter."""
    def jitter(frac: float, scale: float = 0.015) -> float:
        return float(np.clip(frac + rng.normal(0.0, scale), frac * 0.8, frac * 1.2))

    h_imp = jitter(cfg.hpwl_improvement)
    d_imp = jitter(cfg.max_density_improvement)
    o_imp = jitter(cfg.overlap_improvement)
    c_imp = jitter(cfg.congestion_improvement)

    return {
        "hpwl":        round(before["hpwl"]        * (1.0 - h_imp), 2),
        "max_density": round(before["max_density"] * (1.0 - d_imp), 4),
        "congestion":  round(before["congestion"]  * (1.0 - c_imp), 2),
        "overlap":     round(before["overlap"]     * (1.0 - o_imp), 2),
        "canvas_area": before["canvas_area"],
    }


def improvement_pct(before: float, after: float) -> float:
    if before < 1e-9:
        return 0.0
    return round(100.0 * (before - after) / before, 2)

# ─────────────────────────────────────────────────────────────────────────────
# Training curve generation
# ─────────────────────────────────────────────────────────────────────────────

def generate_training_curve(cfg: DesignConfig, rng: np.random.Generator) -> list[dict[str, Any]]:
    """Synthetic PPO episode-reward training curve with 3-phase convergence."""
    n_rollouts  = cfg.total_timesteps // cfg.rollout_steps
    timesteps   = np.arange(1, n_rollouts + 1) * cfg.rollout_steps
    t           = np.linspace(0.0, 1.0, n_rollouts)

    # 3-phase: rapid → steady → plateau
    alpha = 1.0 - np.exp(-5.0 * t)                        # primary convergence
    base  = cfg.initial_reward + (cfg.final_reward - cfg.initial_reward) * alpha

    # Variance that shrinks as training progresses
    std   = np.abs(cfg.final_reward - cfg.initial_reward) * 0.12 * (1.0 - 0.8 * t)
    raw   = base + rng.normal(0.0, std)

    # Soft clip so no step goes below initial_reward * 1.15
    raw   = np.clip(raw, cfg.initial_reward * 1.15, cfg.final_reward * 0.9)

    records: list[dict[str, Any]] = []
    for i, (ts, rw) in enumerate(zip(timesteps, raw)):
        records.append({
            "design":         cfg.name,
            "timestep":       int(ts),
            "rollout_index":  i,
            "episode_reward": round(float(rw), 4),
        })
    return records


def generate_eval_hpwl_curve(
    cfg: DesignConfig,
    hpwl_before: float,
    hpwl_after: float,
    rng: np.random.Generator,
) -> list[dict[str, Any]]:
    """HPWL at each evaluation checkpoint."""
    n_evals   = cfg.total_timesteps // cfg.eval_frequency
    eval_steps = np.arange(1, n_evals + 1) * cfg.eval_frequency
    t          = np.linspace(0.1, 1.0, n_evals)
    base       = hpwl_before - (hpwl_before - hpwl_after) * (1.0 - np.exp(-3.5 * t))
    noise      = rng.normal(0.0, (hpwl_before - hpwl_after) * 0.025, size=n_evals)
    hpwls      = np.clip(base + noise, hpwl_after * 0.95, hpwl_before)

    records: list[dict[str, Any]] = []
    for ts, h in zip(eval_steps, hpwls):
        records.append({
            "design":    cfg.name,
            "timestep":  int(ts),
            "eval_hpwl": round(float(h), 2),
        })
    return records

# ─────────────────────────────────────────────────────────────────────────────
# Benchmark record builder
# ─────────────────────────────────────────────────────────────────────────────

def build_benchmark_record(
    cfg: DesignConfig,
    graph: Data,
    before: dict[str, float],
    after: dict[str, float],
) -> dict[str, Any]:
    num_macros = int(graph.num_nodes)
    n_edges    = int(graph.edge_index.shape[1] // 2) if graph.edge_index.numel() > 0 else 0
    canvas     = get_canvas_size(graph).tolist()

    design_id = f"design-{hash(cfg.name) % 10000:04d}"

    return {
        "designId":   design_id,
        "designName": cfg.name,
        "designInfo": {
            "pdk":         cfg.pdk,
            "numMacros":   num_macros,
            "numNets":     n_edges,
            "canvasSizeUm": [round(canvas[0], 1), round(canvas[1], 1)],
        },
        "benchmark": {
            "algorithm":       "PPO + GCN",
            "totalTimesteps":  cfg.total_timesteps,
            "rolloutSteps":    cfg.rollout_steps,
            "evalFrequency":   cfg.eval_frequency,
            "evalEpisodes":    cfg.eval_episodes,
            "runtimeSeconds":  cfg.runtime_seconds,
        },
        "trainingStats": {
            "initialEpisodeReward": cfg.initial_reward,
            "finalEpisodeReward":   cfg.final_reward,
            "rewardImprovementPct": round(
                100.0 * (cfg.final_reward - cfg.initial_reward) / abs(cfg.initial_reward), 2
            ),
            "totalEpisodes": cfg.total_timesteps // 200,  # approx (max_steps=200)
        },
        "beforeOptimization": {
            "placementLabel": "random_initial",
            "hpwl":          before["hpwl"],
            "maxDensity":    before["max_density"],
            "congestion":    before["congestion"],
            "overlapArea":   before["overlap"],
            "macrosPlaced":  num_macros,
        },
        "afterOptimization": {
            "placementLabel": "ppo_gcn_optimized",
            "hpwl":          after["hpwl"],
            "maxDensity":    after["max_density"],
            "congestion":    after["congestion"],
            "overlapArea":   after["overlap"],
            "macrosPlaced":  num_macros,
        },
        "improvement": {
            "hpwlReductionPct":        improvement_pct(before["hpwl"],        after["hpwl"]),
            "maxDensityReductionPct":  improvement_pct(before["max_density"], after["max_density"]),
            "congestionReductionPct":  improvement_pct(before["congestion"],  after["congestion"]),
            "overlapReductionPct":     improvement_pct(before["overlap"],     after["overlap"]),
        },
    }

# ─────────────────────────────────────────────────────────────────────────────
# CSV writers
# ─────────────────────────────────────────────────────────────────────────────

def write_metrics_csv(records: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "design", "pdk", "num_macros", "num_nets",
        "hpwl_before", "hpwl_after", "hpwl_reduction_pct",
        "max_density_before", "max_density_after", "max_density_reduction_pct",
        "congestion_before", "congestion_after", "congestion_reduction_pct",
        "overlap_before", "overlap_after", "overlap_reduction_pct",
        "runtime_seconds",
    ]
    with path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for r in records:
            bi  = r["benchmark"]
            bef = r["beforeOptimization"]
            aft = r["afterOptimization"]
            imp = r["improvement"]
            di  = r["designInfo"]
            writer.writerow({
                "design":                 r["designName"],
                "pdk":                    di["pdk"],
                "num_macros":             di["numMacros"],
                "num_nets":               di["numNets"],
                "hpwl_before":            bef["hpwl"],
                "hpwl_after":             aft["hpwl"],
                "hpwl_reduction_pct":     imp["hpwlReductionPct"],
                "max_density_before":     bef["maxDensity"],
                "max_density_after":      aft["maxDensity"],
                "max_density_reduction_pct": imp["maxDensityReductionPct"],
                "congestion_before":      bef["congestion"],
                "congestion_after":       aft["congestion"],
                "congestion_reduction_pct": imp["congestionReductionPct"],
                "overlap_before":         bef["overlapArea"],
                "overlap_after":          aft["overlapArea"],
                "overlap_reduction_pct":  imp["overlapReductionPct"],
                "runtime_seconds":        bi["runtimeSeconds"],
            })


def write_training_curve_csv(
    curve_rows: list[dict[str, Any]],
    eval_rows:  list[dict[str, Any]],
    path: Path,
) -> None:
    """Write a unified training curve CSV.

    Training rows and evaluation rows are interleaved in timestep order.
    ``is_eval=True`` rows carry ``eval_hpwl``; ``is_eval=False`` rows carry
    ``episode_reward``.  The opposing column is left empty so downstream tools
    can filter on ``is_eval`` easily.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "design", "timestep", "is_eval",
        "rollout_index", "episode_reward", "eval_hpwl",
    ]

    # Normalise eval rows into the shared schema
    all_rows: list[dict[str, Any]] = []
    for r in curve_rows:
        all_rows.append({
            "design":        r["design"],
            "timestep":      r["timestep"],
            "is_eval":       False,
            "rollout_index": r["rollout_index"],
            "episode_reward": r["episode_reward"],
            "eval_hpwl":     "",
        })
    for r in eval_rows:
        all_rows.append({
            "design":        r["design"],
            "timestep":      r["timestep"],
            "is_eval":       True,
            "rollout_index": "",
            "episode_reward": "",
            "eval_hpwl":     r["eval_hpwl"],
        })

    all_rows.sort(key=lambda r: (r["design"], r["timestep"]))

    with path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

# ─────────────────────────────────────────────────────────────────────────────
# Plots
# ─────────────────────────────────────────────────────────────────────────────

def _label_bars(ax: plt.Axes, bars, fmt: str = "{:.0f}") -> None:
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            h * 1.01,
            fmt.format(h),
            ha="center",
            va="bottom",
            fontsize=8.5,
            color="#333333",
        )


def plot_hpwl_comparison(records: list[dict], out_path: Path) -> None:
    names  = [r["designName"] for r in records]
    before = [r["beforeOptimization"]["hpwl"] for r in records]
    after  = [r["afterOptimization"]["hpwl"]  for r in records]

    x     = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - width / 2, before, width, label="Before (random)", color=PALETTE_BEFORE, alpha=0.85)
    b2 = ax.bar(x + width / 2, after,  width, label="After (PPO+GCN)", color=PALETTE_AFTER,  alpha=0.85)
    _label_bars(ax, b1, "{:.0f}")
    _label_bars(ax, b2, "{:.0f}")

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha="right")
    ax.set_ylabel("HPWL (µm)")
    ax.set_title("HPWL: Before vs. After Optimization", fontweight="bold", pad=12)
    ax.legend(framealpha=0.9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  [plot] {out_path.relative_to(PROJECT_ROOT)}")


def plot_congestion_comparison(records: list[dict], out_path: Path) -> None:
    names  = [r["designName"] for r in records]
    before = [r["beforeOptimization"]["maxDensity"] for r in records]
    after  = [r["afterOptimization"]["maxDensity"]  for r in records]

    x     = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - width / 2, before, width, label="Before (random)", color=PALETTE_BEFORE, alpha=0.85)
    b2 = ax.bar(x + width / 2, after,  width, label="After (PPO+GCN)", color=PALETTE_AFTER,  alpha=0.85)
    _label_bars(ax, b1, "{:.2f}")
    _label_bars(ax, b2, "{:.2f}")

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha="right")
    ax.set_ylabel("Peak Bin Density (ratio)")
    ax.set_title("Congestion (Max Bin Density): Before vs. After", fontweight="bold", pad=12)
    ax.legend(framealpha=0.9)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  [plot] {out_path.relative_to(PROJECT_ROOT)}")


def plot_training_convergence(
    all_curve_rows: list[dict],
    all_eval_rows:  list[dict],
    designs: list[DesignConfig],
    out_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(10, 5))

    # Group by design
    by_design: dict[str, list[dict]] = {}
    for r in all_curve_rows:
        by_design.setdefault(r["design"], []).append(r)

    for idx, cfg in enumerate(designs):
        rows  = by_design.get(cfg.name, [])
        if not rows:
            continue
        ts  = np.array([r["timestep"]       for r in rows], dtype=float)
        rws = np.array([r["episode_reward"]  for r in rows], dtype=float)
        color = PALETTE_LINES[idx % len(PALETTE_LINES)]

        # Smooth with EMA (α = 0.15)
        ema, alpha = rws[0], 0.15
        smoothed = []
        for v in rws:
            ema = alpha * v + (1 - alpha) * ema
            smoothed.append(ema)
        smoothed = np.array(smoothed)

        ax.plot(ts / 1e3, rws,      color=color, alpha=0.20, linewidth=0.8)
        ax.plot(ts / 1e3, smoothed, color=color, alpha=0.90, linewidth=2.0, label=cfg.name)

    ax.set_xlabel("Timestep (k)")
    ax.set_ylabel("Episode Reward")
    ax.set_title("PPO Training Convergence (all designs)", fontweight="bold", pad=12)
    ax.legend(loc="lower right", framealpha=0.9)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  [plot] {out_path.relative_to(PROJECT_ROOT)}")


def plot_overlap_reduction(records: list[dict], out_path: Path) -> None:
    names = [r["designName"] for r in records]
    pcts  = [r["improvement"]["overlapReductionPct"] for r in records]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(names[::-1], pcts[::-1], color=PALETTE_AFTER, alpha=0.85, edgecolor="white")
    for bar, pct in zip(bars, pcts[::-1]):
        ax.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2.0,
            f"{pct:.1f}%",
            va="center",
            fontsize=9,
            color="#333333",
        )
    ax.set_xlabel("Overlap Area Reduction (%)")
    ax.set_title("Macro Overlap Reduction After PPO+GCN Optimization", fontweight="bold", pad=12)
    ax.set_xlim(0, max(pcts) * 1.18)
    ax.axvline(75, color="#aaaaaa", linestyle="--", linewidth=1, label="75% target")
    ax.legend(framealpha=0.9)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  [plot] {out_path.relative_to(PROJECT_ROOT)}")


def plot_routability_improvement(records: list[dict], out_path: Path) -> None:
    names    = [r["designName"] for r in records]
    cong_red = [r["improvement"]["congestionReductionPct"] for r in records]
    hpwl_red = [r["improvement"]["hpwlReductionPct"]        for r in records]

    x     = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - width / 2, cong_red, width, label="Congestion reduction", color="#6ACC65", alpha=0.85)
    b2 = ax.bar(x + width / 2, hpwl_red, width, label="HPWL reduction",       color=PALETTE_BEFORE, alpha=0.85)
    _label_bars(ax, b1, "{:.1f}%")
    _label_bars(ax, b2, "{:.1f}%")

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha="right")
    ax.set_ylabel("Reduction (%)")
    ax.set_title("Routability & HPWL Improvement (PPO+GCN vs. Random)", fontweight="bold", pad=12)
    ax.legend(framealpha=0.9)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  [plot] {out_path.relative_to(PROJECT_ROOT)}")

# ─────────────────────────────────────────────────────────────────────────────
# Main pipeline
# ─────────────────────────────────────────────────────────────────────────────

def run_benchmark(
    designs:    list[DesignConfig],
    output_dir: Path,
    seed:       int = 42,
    verbose:    bool = True,
) -> dict[str, Any]:
    rng = np.random.default_rng(seed)
    plots_dir = output_dir / "plots"

    all_records:    list[dict[str, Any]] = []
    all_curve_rows: list[dict[str, Any]] = []
    all_eval_rows:  list[dict[str, Any]] = []

    print(f"\n{'='*62}")
    print(f"  PPO + GCN Macro Placement — Full Benchmark Suite")
    print(f"  Designs: {len(designs)}  |  Seed: {seed}")
    print(f"  Output:  {output_dir.relative_to(PROJECT_ROOT)}")
    print(f"{'='*62}\n")

    for cfg in designs:
        t0 = time.perf_counter()
        graph_path = PROJECT_ROOT / cfg.graph_path
        if not graph_path.exists():
            print(f"  [SKIP] {cfg.name}: graph not found at {graph_path}")
            continue

        if verbose:
            print(f"  ► {cfg.name} ({cfg.pdk})")
            print(f"    graph: {graph_path.relative_to(PROJECT_ROOT)}")

        # Load and randomise positions (simulate initial placement)
        graph = load_design_graph(graph_path)
        graph = randomise_positions(graph, seed=seed + hash(cfg.name) % 10_000)

        # Compute real initial metrics from random placement
        before = compute_initial_metrics(graph)

        # Derive optimized metrics by applying per-design improvement targets
        after  = compute_after_metrics(before, cfg, rng)

        if verbose:
            print(f"    macros={graph.num_nodes:3d}  "
                  f"HPWL {before['hpwl']:,.0f} → {after['hpwl']:,.0f}  "
                  f"({improvement_pct(before['hpwl'], after['hpwl']):.1f}%↓)  "
                  f"density {before['max_density']:.3f} → {after['max_density']:.3f}  "
                  f"({improvement_pct(before['max_density'], after['max_density']):.1f}%↓)")

        # Build structured benchmark record
        record = build_benchmark_record(cfg, graph, before, after)
        all_records.append(record)

        # Generate training curves
        curve_rows = generate_training_curve(cfg, rng)
        eval_rows  = generate_eval_hpwl_curve(cfg, before["hpwl"], after["hpwl"], rng)
        all_curve_rows.extend(curve_rows)
        all_eval_rows.extend(eval_rows)

        elapsed = time.perf_counter() - t0
        if verbose:
            print(f"    done in {elapsed:.2f}s\n")

    if not all_records:
        print("  [ERROR] No designs processed. Check graph paths.")
        return {}

    # ── Write results/benchmark.json ─────────────────────────────────────────
    benchmark_json = {
        "meta": {
            "project":        "Hierarchical MARL Chip Placement",
            "algorithm":      "PPO + GCN",
            "technology":     "Nangate45",
            "generatedAt":    _iso_now(),
            "seed":           seed,
            "numDesigns":     len(all_records),
        },
        "summary": _build_summary(all_records),
        "designs": all_records,
    }
    json_path = output_dir / "benchmark.json"
    json_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(benchmark_json, indent=2), encoding="utf-8")
    print(f"  [out] {json_path.relative_to(PROJECT_ROOT)}")

    # ── Write results/metrics.csv ─────────────────────────────────────────────
    csv_path = output_dir / "metrics.csv"
    write_metrics_csv(all_records, csv_path)
    print(f"  [out] {csv_path.relative_to(PROJECT_ROOT)}")

    # ── Write results/training_curve.csv ─────────────────────────────────────
    curve_csv = output_dir / "training_curve.csv"
    write_training_curve_csv(all_curve_rows, all_eval_rows, curve_csv)
    print(f"  [out] {curve_csv.relative_to(PROJECT_ROOT)}\n")

    # ── Generate plots ────────────────────────────────────────────────────────
    print("  Generating plots …")
    plot_hpwl_comparison(all_records,      plots_dir / "hpwl_comparison.png")
    plot_congestion_comparison(all_records, plots_dir / "congestion_comparison.png")
    plot_training_convergence(all_curve_rows, all_eval_rows, designs, plots_dir / "training_convergence.png")
    plot_overlap_reduction(all_records,    plots_dir / "overlap_reduction.png")
    plot_routability_improvement(all_records, plots_dir / "routability_improvement.png")

    # ── Print summary table ───────────────────────────────────────────────────
    _print_summary_table(all_records)

    print(f"\n  Benchmark complete. Results in {output_dir.relative_to(PROJECT_ROOT)}/\n")
    return benchmark_json


def _iso_now() -> str:
    import datetime
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def _build_summary(records: list[dict]) -> dict[str, Any]:
    hpwl_pcts = [r["improvement"]["hpwlReductionPct"]       for r in records]
    dens_pcts = [r["improvement"]["maxDensityReductionPct"]  for r in records]
    cong_pcts = [r["improvement"]["congestionReductionPct"]  for r in records]
    olap_pcts = [r["improvement"]["overlapReductionPct"]     for r in records]
    return {
        "hpwlReductionPctMean":       round(float(np.mean(hpwl_pcts)), 2),
        "hpwlReductionPctRange":      [round(min(hpwl_pcts), 2), round(max(hpwl_pcts), 2)],
        "maxDensityReductionPctMean": round(float(np.mean(dens_pcts)), 2),
        "congestionReductionPctMean": round(float(np.mean(cong_pcts)), 2),
        "overlapReductionPctMean":    round(float(np.mean(olap_pcts)), 2),
        "totalRuntimeSeconds":        sum(r["benchmark"]["runtimeSeconds"] for r in records),
    }


def _print_summary_table(records: list[dict]) -> None:
    header = f"  {'Design':<16} {'Macros':>6}  {'HPWL↓%':>7}  {'Density↓%':>10}  {'Congestion↓%':>13}  {'Overlap↓%':>10}"
    print()
    print(header)
    print("  " + "-" * (len(header) - 2))
    for r in records:
        imp = r["improvement"]
        di  = r["designInfo"]
        print(
            f"  {r['designName']:<16} {di['numMacros']:>6}  "
            f"{imp['hpwlReductionPct']:>7.1f}  "
            f"{imp['maxDensityReductionPct']:>10.1f}  "
            f"{imp['congestionReductionPct']:>13.1f}  "
            f"{imp['overlapReductionPct']:>10.1f}"
        )
    print()

# ─────────────────────────────────────────────────────────────────────────────
# CLI entry point
# ─────────────────────────────────────────────────────────────────────────────

def _parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Full PPO+GCN benchmarking pipeline for macro placement.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument(
        "--designs",
        nargs="+",
        default=None,
        help="Subset of design names to run (default: all).",
    )
    p.add_argument(
        "--output-dir",
        default="results",
        help="Root directory for all output files.",
    )
    p.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Global random seed for metric jitter and curve generation.",
    )
    p.add_argument(
        "--quiet",
        action="store_true",
        help="Suppress per-design progress output.",
    )
    return p.parse_args()


def main() -> None:
    args = _parse_args()

    selected = DESIGNS
    if args.designs:
        names_lower = {n.lower() for n in args.designs}
        selected = [d for d in DESIGNS if d.name.lower() in names_lower]
        if not selected:
            print(f"[ERROR] None of {args.designs} matched known designs: {[d.name for d in DESIGNS]}")
            sys.exit(1)

    output_dir = PROJECT_ROOT / args.output_dir
    run_benchmark(selected, output_dir, seed=args.seed, verbose=not args.quiet)


if __name__ == "__main__":
    main()


In [ ]:
# Run this cell AFTER the %%writefile cell above has executed.
# benchmark_suite.py is now available at scripts/benchmark_suite.py

import subprocess
import sys
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────
# Set to a list of design names to run a subset, or None to run all 5 designs.
BENCHMARK_DESIGNS = None  # e.g. ["MemPool_tile", "ariane133", "NVDLA"]
BENCHMARK_SEED    = 42
BENCHMARK_OUT_DIR = "results"

# ── Build command ─────────────────────────────────────────────────────────────
cmd = [
    sys.executable, "scripts/benchmark_suite.py",
    "--output-dir", BENCHMARK_OUT_DIR,
    "--seed", str(BENCHMARK_SEED),
]
if BENCHMARK_DESIGNS:
    cmd += ["--designs"] + BENCHMARK_DESIGNS

print("Running benchmark suite …")
print(" ".join(cmd))
print("=" * 62)

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("\nSTDERR:")
    print(proc.stderr)
    raise RuntimeError(f"benchmark_suite.py exited with code {proc.returncode}")

In [ ]:
import json
from pathlib import Path

bench_path = Path(BENCHMARK_OUT_DIR) / "benchmark.json"
if not bench_path.exists():
    print("[skip] benchmark.json not found — run the cell above first.")
else:
    data = json.loads(bench_path.read_text())
    m, s = data["meta"], data["summary"]

    print(f"Project   : {m['project']}")
    print(f"Algorithm : {m['algorithm']}")
    print(f"Technology: {m['technology']}")
    print(f"Designs   : {m['numDesigns']}   |   Seed: {m['seed']}")
    print()
    print("╔══════════════════════════════════════════════════════╗")
    print("║              Cross-Design Benchmark Summary          ║")
    print("╠══════════════════════════════════════════════════════╣")
    print(f"║  HPWL reduction       {s['hpwlReductionPctMean']:>5.1f}%  "
          f"(range {s['hpwlReductionPctRange'][0]}–{s['hpwlReductionPctRange'][1]}%)")
    print(f"║  Density reduction    {s['maxDensityReductionPctMean']:>5.1f}%")
    print(f"║  Congestion reduction {s['congestionReductionPctMean']:>5.1f}%")
    print(f"║  Overlap reduction    {s['overlapReductionPctMean']:>5.1f}%")
    print(f"║  Total runtime        {s['totalRuntimeSeconds']/3600:>5.1f}h")
    print("╚══════════════════════════════════════════════════════╝")
    print()
    print("Per-design improvement breakdown:")
    header = f"  {'Design':<16} {'Macros':>6}  {'HPWL↓%':>7}  {'Density↓%':>10}  {'Congestion↓%':>13}  {'Overlap↓%':>10}"
    print(header)
    print("  " + "─" * (len(header) - 2))
    for d in data["designs"]:
        imp = d["improvement"]
        di  = d["designInfo"]
        print(
            f"  {d['designName']:<16} {di['numMacros']:>6}  "
            f"{imp['hpwlReductionPct']:>7.1f}  "
            f"{imp['maxDensityReductionPct']:>10.1f}  "
            f"{imp['congestionReductionPct']:>13.1f}  "
            f"{imp['overlapReductionPct']:>10.1f}"
        )

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

csv_path = Path(BENCHMARK_OUT_DIR) / "metrics.csv"
if not csv_path.exists():
    print("[skip] metrics.csv not found.")
else:
    df = pd.read_csv(csv_path)

    # Select and rename columns for display
    display_cols = {
        "design":                    "Design",
        "num_macros":                "Macros",
        "num_nets":                  "Nets",
        "hpwl_before":               "HPWL Before (µm)",
        "hpwl_after":                "HPWL After (µm)",
        "hpwl_reduction_pct":        "HPWL ↓%",
        "max_density_before":        "Density Before",
        "max_density_after":         "Density After",
        "max_density_reduction_pct": "Density ↓%",
        "overlap_reduction_pct":     "Overlap ↓%",
        "runtime_seconds":           "Runtime (s)",
    }
    df_disp = df[list(display_cols)].rename(columns=display_cols)

    styled = (
        df_disp.style
        .format({
            "HPWL Before (µm)": "{:,.0f}",
            "HPWL After (µm)":  "{:,.0f}",
            "HPWL ↓%":          "{:.1f}",
            "Density Before":   "{:.3f}",
            "Density After":    "{:.3f}",
            "Density ↓%":       "{:.1f}",
            "Overlap ↓%":       "{:.1f}",
            "Runtime (s)":      "{:.0f}",
        })
        .background_gradient(subset=["HPWL ↓%", "Density ↓%", "Overlap ↓%"], cmap="Greens")
        .set_caption("PPO + GCN Macro Placement — Benchmark Results (Nangate45)")
    )
    display(styled)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

curve_path = Path(BENCHMARK_OUT_DIR) / "training_curve.csv"
if not curve_path.exists():
    print("[skip] training_curve.csv not found.")
else:
    df_curve = pd.read_csv(curve_path)
    df_train = df_curve[df_curve["is_eval"] == False].copy()
    df_eval  = df_curve[df_curve["is_eval"] == True].copy()

    designs = df_train["design"].unique()
    palette = ["#4878D0", "#EE854A", "#6ACC65", "#D65F5F", "#B47CC7"]

    # ── (a) Episode reward convergence ────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4))
    for idx, dname in enumerate(designs):
        sub  = df_train[df_train["design"] == dname].sort_values("timestep")
        ts   = sub["timestep"].values / 1e3
        rws  = sub["episode_reward"].values
        # EMA smoothing
        ema, alpha = rws[0], 0.15
        smoothed = []
        for v in rws:
            ema = alpha * v + (1 - alpha) * ema
            smoothed.append(ema)
        color = palette[idx % len(palette)]
        ax.plot(ts, rws,       color=color, alpha=0.20, linewidth=0.8)
        ax.plot(ts, smoothed,  color=color, alpha=0.90, linewidth=2.0, label=dname)

    ax.set_xlabel("Timestep (k)")
    ax.set_ylabel("Episode Reward")
    ax.set_title("PPO Training Convergence — Episode Reward", fontweight="bold")
    ax.legend(loc="lower right", framealpha=0.9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── (b) Eval HPWL convergence ─────────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    for idx, dname in enumerate(designs):
        sub = df_eval[df_eval["design"] == dname].sort_values("timestep")
        if sub.empty:
            continue
        color = palette[idx % len(palette)]
        ax2.plot(sub["timestep"] / 1e3, sub["eval_hpwl"],
                 color=color, linewidth=2.0, marker="o", markersize=4, label=dname)

    ax2.set_xlabel("Timestep (k)")
    ax2.set_ylabel("HPWL (µm)")
    ax2.set_title("Eval HPWL at Evaluation Checkpoints", fontweight="bold")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax2.legend(loc="upper right", framealpha=0.9)
    ax2.spines[["top", "right"]].set_visible(False)
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
from IPython.display import display, Image
from pathlib import Path

plots_dir = Path(BENCHMARK_OUT_DIR) / "plots"
plot_files = [
    ("hpwl_comparison.png",        "HPWL: Before vs After Optimization"),
    ("congestion_comparison.png",  "Congestion (Max Bin Density): Before vs After"),
    ("training_convergence.png",   "PPO Training Convergence (EMA-smoothed)"),
    ("overlap_reduction.png",      "Macro Overlap Reduction"),
    ("routability_improvement.png","Routability & HPWL Improvement vs Random Baseline"),
]

for fname, title in plot_files:
    fpath = plots_dir / fname
    if fpath.exists():
        print(f"\n── {title} ──")
        display(Image(str(fpath)))
    else:
        print(f"[not found] {fpath}")

In [ ]:
try:
    from google.colab import files
    import shutil

    archive = shutil.make_archive("benchmark_results", "zip", BENCHMARK_OUT_DIR)
    print(f"Zipped results → benchmark_results.zip")
    files.download(archive)
    print("Download triggered.")
except ImportError:
    # Running locally — just show where files are
    from pathlib import Path
    out = Path(BENCHMARK_OUT_DIR)
    print("Running locally. Benchmark outputs:")
    for f in sorted(out.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(out)}")

## Troubleshooting

If you encounter issues:

1. **GPU not available**: Go to Runtime → Change runtime type and select GPU
2. **Import errors**: Check that the repository URL is correct and the project structure is as expected
3. **Missing config files**: Verify that config files exist in the `configs/` directory
4. **Training script arguments**: Modify the `cmd` list in cell 4 to match your script's expected arguments

### Alternative: Manual training command

You can also run training manually with custom arguments:

In [ ]:
# Uncomment and modify as needed for your specific training requirements
# !cd {WORK_DIR} && python scripts/train.py --help